# Lab F - Manhattan and Chebyshev distances
For points p, q: **Manhattan (L1)** = sum |p_i - q_i|; **Euclidean (L2)** = sqrt(sum (p_i-q_i)^2); **Chebyshev (L-inf)** = max |p_i - q_i|.

In [ ]:
import sys; sys.path.insert(0, '../../src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
from scipy.spatial import distance
p, q = np.array([1, 2, 3]), np.array([4, 6, 3])
print('diff =', p - q)
print('Manhattan = |1-4|+|2-6|+|3-3| =', np.abs(p-q).sum(), ' (scipy', distance.cityblock(p, q), ')')
print('Euclidean = sqrt(9+16+0) =', np.sqrt(((p-q)**2).sum()), ' (scipy', distance.euclidean(p, q), ')')
print('Chebyshev = max(3,4,0) =', np.abs(p-q).max(), ' (scipy', distance.chebyshev(p, q), ')')

### Unit balls (all points at distance 1 from the origin)

In [ ]:
t = np.linspace(0, 2*np.pi, 400); fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(np.cos(t), np.sin(t), label='Euclidean (circle)')
ax.plot([1,0,-1,0,1], [0,1,0,-1,0], label='Manhattan (diamond)'); ax.plot([1,-1,-1,1,1], [1,1,-1,-1,1], label='Chebyshev (square)')
ax.set_aspect('equal'); ax.legend(fontsize=7, loc='lower right'); plt.show()

### Bridge to ML: kNN on Titanic with each metric (same split per seed, k=5, 5 seeds)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
from arena.evaluate import split
from arena.preprocess import build_preprocessor
rows = []
for seed in range(5):
    Xtr, Xte, ytr, yte = split('clf', seed); pre, _ = build_preprocessor(Xtr); Ztr = pre.fit_transform(Xtr); Zte = pre.transform(Xte)
    for metric in ['manhattan', 'euclidean', 'chebyshev']:
        pred = KNeighborsClassifier(5, metric=metric).fit(Ztr, ytr).predict(Zte)
        rows.append(dict(metric=metric, seed=seed, accuracy=accuracy_score(yte, pred), f1_macro=f1_score(yte, pred, average='macro')))
r = pd.DataFrame(rows).groupby('metric')[['accuracy', 'f1_macro']].agg(['mean', 'std']).round(3); r

Differences between metrics are small relative to the seed-to-seed std on this small test set; the choice matters more in high dimensions and for data with outliers.